# Exploracion inicial de corpus y su metadata

## importamos los datos y librerias

In [ ]:
from utils import load_corpus_raw

df = load_corpus_raw()
print(df.shape)
df.head(3)

## Tipos de datos

In [ ]:
df.dtypes

## Identificación de datos nulos en el corpus

In [ ]:
df.isna().sum()

## Identificación de textos vacios

In [ ]:
(df["texto"].astype(str).str.strip() == "").sum()

## Trasnformación de columna fecha a tipo DateTime

In [ ]:
import pandas as pd

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["fecha"].isna().sum()  # fechas que no pudieron parsearse

In [ ]:
df["fecha"].dt.time.value_counts().head()

## Duplicidad de datos

In [ ]:
df["id"].duplicated().sum(), df["texto"].duplicated().sum()

In [ ]:
dups = df[df["texto"].duplicated(keep=False)].sort_values("texto")

In [ ]:
dups[["id", "fuente", "fecha"]].head(20)

In [ ]:
dups["texto"].str.split().str.len().describe()

### Validación de fuentes de textos duplicados

In [ ]:
g = dups.groupby("texto", observed=True).agg(
    n=("id", "size"),
    fuentes=("fuente", lambda s: sorted(set(s))),
    fechas=("fecha", lambda s: sorted(s.dt.date.unique())),
)
g["fuentes"].astype(str).value_counts()

In [ ]:
df["fecha"].dt.day.value_counts().head(10)

In [ ]:
df["fecha"].dt.day.value_counts().sort_index().head(3)

In [ ]:
par = dups["texto"].iloc[0]
print(par[:400])

In [ ]:
dups[dups["texto"] == par][["id", "fuente", "fecha"]]

### Se deciden eliminar los textos repetidos

In [ ]:
df_raw = load_corpus_raw()

norm = df_raw["texto"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

df = df_raw[~norm.duplicated()].reset_index(drop=True)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

print(len(df_raw), "→", len(df))  # 91749 → 91585

In [ ]:
assert len(df) == 91585, f"esperaba 91585, hay {len(df)}"
assert df["texto"].duplicated().sum() == 0
assert norm[~norm.duplicated()].duplicated().sum() == 0
assert df["fecha"].isna().sum() == 0
assert df.isna().sum().sum() == 0
print("OK:", len(df), "documentos únicos")

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

In [ ]:
n_pal = df["texto"].str.split().str.len()
print(n_pal.sum(), n_pal.mean())

solo_pal = df["texto"].str.count(r"\b\w+\b")
print(solo_pal.sum())

In [ ]:
solo_pal.mean()

In [ ]:
df["fuente"].value_counts()

In [ ]:
df["fuente"].value_counts(normalize=True).mul(100).round(1)


In [ ]:
solo_pal.describe()

In [ ]:
df["n_palabras"] = solo_pal

In [ ]:
df["n_palabras"] = solo_pal
cortos = df[df["n_palabras"] <= 84]
print(len(cortos))

for t in cortos["texto"].sample(5, random_state=42):
    print(t[:300], "\n---")

In [ ]:
t = cortos["texto"].sample(1, random_state=42).iloc[0]
print(len(t.split()), "palabras")
print(t)

In [ ]:
df["texto"].str.contains(r"VER GRAFICO|VER FOTO|VER TABLA", case=False).sum()

In [ ]:
por_anio = df["fecha"].dt.year.value_counts().sort_index()
print(por_anio)

In [ ]:
import json

import matplotlib.pyplot as plt

from utils import results_dir

RESULTS = results_dir()

ax = por_anio.plot.bar(figsize=(9, 3.2), width=0.85, color="#444")
ax.set_xlabel("Año")
ax.set_ylabel("Documentos")
plt.tight_layout()
plt.savefig(RESULTS / "docs_por_anio.png", dpi=300, bbox_inches="tight")
plt.show()

stats = {
    "documentos_original": len(df_raw),
    "documentos_unicos": len(df),
    "duplicados_exactos": 123,
    "duplicados_normalizados": 164,
    "palabras": int(solo_pal.sum()),
    "promedio_palabras": float(solo_pal.mean()),
    "tokens_split": int(n_pal.sum()),
    "marcadores_maquetacion": 408,
}
(RESULTS / "corpus_stats.json").write_text(json.dumps(stats, indent=2))

por_anio.to_frame("documentos").to_csv(RESULTS / "docs_por_anio.csv")
df["fuente"].value_counts().to_frame("documentos").to_csv(
    RESULTS / "docs_por_fuente.csv"
)
solo_pal.describe().to_frame("palabras").to_csv(RESULTS / "longitudes.csv")